# heritage3d-retrieval — Quickstart

A runnable walkthrough for installing, configuring and using the **heritage3d-retrieval** toolkit:
harvest cultural-heritage records from several providers into a single SQLite database, then (optionally) enrich and convert.

**Prerequisites:** Python ≥ 3.10, and this notebook placed *inside the unzipped `heritage3d-retrieval` repository folder* (next to `pyproject.toml`).
Run the cells top to bottom.

## 1. Locate the repository
Make sure the working directory is the repo root (where `pyproject.toml` lives).

In [1]:
import os, pathlib

# If pyproject.toml isn't here, set REPO to the repo path and re-run.
REPO = pathlib.Path.cwd()
if not (REPO / 'pyproject.toml').exists():
    # common case: the unzipped folder sits next to this notebook
    cand = REPO / 'heritage3d-retrieval'
    REPO = cand if (cand / 'pyproject.toml').exists() else REPO
os.chdir(REPO)
assert (REPO / 'pyproject.toml').exists(), f'pyproject.toml not found in {REPO} — set REPO manually.'
print('Repository:', REPO)

Repository: /Users/muenster/Nextcloud/Cloud Digital Humanities (2)/DH Professorship Project Management/Projekte/EU-2024-DEP-3DBigDataSpace/Deliverables/D2.1/heritage3d-retrieval


## 2. Install the package
Core install. Add extras if you need them: `[europeana]`, `[enrich]` (geocoding/NER/LLM), `[convert]` (3D), or `[all]`.

In [2]:
%pip install -e .
# %pip install -e ".[europeana]"   # Europeana helpers
# %pip install -e ".[all]"          # everything + dev tools

Obtaining file:///Users/muenster/Nextcloud/Cloud%20Digital%20Humanities%20%282%29/DH%20Professorship%20Project%20Management/Projekte/EU-2024-DEP-3DBigDataSpace/Deliverables/D2.1/heritage3d-retrieval
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for heritage3d-retrieval (pyproject.toml) ... done
  Created wheel for heritage3d-retrieval: filename=heritage3d_retrieval-0.1.0-0.editable-py3-none-any.whl size=4128 sha256=d09a558003ea093f1da715d64c3750a6301204d420223dc86f6d18231f25fce0
  Stored in directory: /private/var/folders/zl/dt4m3cgd0y53blb7fr64gkv40000gn/T/pip-ephem-wheel-cache-4kf27h19/wheels/47/9f/8f/498e39aca1c36a8585224246ed45bba1d89a21d6f797e61bcf
Successfully built heritage3d-retrieval
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [heritage3d-retrieval]
Note: you may need to restart the kernel to u

## 3. Configure credentials
Nothing is hard-coded — every key is read from the environment (optionally a `.env` file).
Only set the keys for the sources you'll actually use. Below we enter the Europeana key securely (no key is stored in the notebook).

In [1]:
import os, getpass, shutil, pathlib

# Create a .env from the template the first time (you can edit it later).
if not pathlib.Path('.env').exists():
    shutil.copy('.env.example', '.env')
    print('Created .env from .env.example — edit it to add your keys.')

# For this session, set the Europeana key in the environment if not already present.
if not os.environ.get('EUROPEANA_API_KEY'):
    os.environ['EUROPEANA_API_KEY'] = getpass.getpass('Europeana API key (free at pro.europeana.eu/page/get-api): ')

# Optional: keep the local database inside the repo folder
os.environ.setdefault('HERITAGE3D_DB', 'heritage3d.sqlite3')
print('Configured. DB path =', os.environ['HERITAGE3D_DB'])

Configured. DB path = heritage3d.sqlite3


## 4. Initialise the database
Creates the SQLite schema (an `object` table). Only needed once.

In [2]:
from heritage3d.config import get_settings
from heritage3d.db import init_db

settings = get_settings()
init_db(settings.database_path)
print('Initialised database at:', settings.database_path)

Initialised database at: heritage3d.sqlite3


## 5. Harvest a small sample (Europeana)
`ingest()` pages through the provider, normalises records and upserts them (idempotent — re-running refreshes, never duplicates).
We use a small `limit` here so it finishes quickly.

In [4]:
from heritage3d.sources import europeana

written = europeana.ingest('building', limit=20, settings=settings)
print(f'Ingested {written} Europeana records.')

HTTPError: 400 Client Error: Bad Request for url: https://api.europeana.eu/record/v2/search.json?wskey=ouselgeniou%E2%89%88&query=building&qf=TYPE%3AIMAGE&rows=100&cursor=%2A&profile=rich&media=true

## 6. Inspect the database
Read the `object` table back and look at what was stored.

In [ ]:
import sqlite3, pandas as pd

con = sqlite3.connect(settings.database_path)
total = pd.read_sql_query('SELECT COUNT(*) AS rows FROM object', con)
by_source = pd.read_sql_query('SELECT source, COUNT(*) AS n FROM object GROUP BY source', con)
sample = pd.read_sql_query('SELECT source, uid, name, edmIsShownBy FROM object LIMIT 5', con)
con.close()

print('Total rows:', int(total['rows'][0]))
display(by_source)
display(sample)

## 7. Other sources
Each provider follows the same `ingest()` pattern. Some need their own credential (set it in `.env` / the environment first).
Uncomment to try.

In [ ]:
from heritage3d.sources import smithsonian, oai_pmh, scan_the_world

#smithsonian.ingest(limit=50, settings=settings)                 # Smithsonian 3D (GLB)
#oai_pmh.ingest('nfdi4ing', limit=50, settings=settings)         # generic OAI-PMH (preset)
#scan_the_world.ingest('cathedral', limit=50, settings=settings) # needs MYMINIFACTORY_TOKEN

RuntimeError: Missing required credential 'myminifactory_token'. Set the MYMINIFACTORY_TOKEN environment variable (see .env.example).

## 8. Command-line interface (alternative to the library)
Everything above is also available as a CLI once the package is installed.

In [ ]:
!heritage3d --help

## 9. (Optional) Run the test suite
Confirms the install is healthy. No network or API keys required.

In [ ]:
%pip install -e ".[dev]" -q
!pytest -q

---
### Notes
- If a call raises a *missing credential* error, it names the environment variable to add to `.env`.
- Enrichment (`heritage3d.enrich.*`) and 3D conversion (`heritage3d.convert.*`) need the `[enrich]` / `[convert]` extras and, in some cases, paid services or a GPU.
- The database is a single portable file (`heritage3d.sqlite3`); back it up by copying it.